# Lesson 27 Lab — Automated Experiment Management and Reproducible Pruning Records

**Puzzle:** Which fields make a pruning mask reproducible by another person?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

A sparsity number cannot identify a run. Reproduction needs data and model revisions, seed, score rule, tie behavior, target, mask bytes or hash, optimizer/recovery schedule, software, hardware, export command, and measured artifacts. Tracking systems help only when these fields are logged.


## 0. Predict before running

1. Predict which hashes match across identical-seed runs.
2. Predict whether a different seed can preserve sparsity while changing the mask.
3. List the minimum fields another machine needs to repeat the result.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A deterministic pruning function is executed twice with one seed and once with another. Configuration JSON, weight initialization, mask, output, and SHA-256 digests are compared in a small run registry.

- Sparsity equality is weaker than mask identity.
- Canonical configuration and binary artifacts need separate hashes.
- A tracking UI cannot compensate for missing provenance fields.


## 2. Derive the mechanism

Random seed controls initialization and sampled data, but deterministic algorithms and stable ordering also matter. Hashing canonical JSON catches configuration drift; hashing contiguous mask bytes identifies the exact support. A code commit and environment complete the provenance. Reproducing the same global sparsity with a different mask is not the same experiment.

### Mechanism at a glance

```mermaid
flowchart LR
  I["commit + model + data + env + seed"] --> R["immutable run manifest"]
  R --> P["pruning and recovery stages"]
  P --> A["checkpoints + masks + metrics"]
  A --> E["export + runtime evidence"]
  E --> C["content hashes and final decision"]
  C --> X["independent reproduction run"]
  X --> G{"manifest tolerances pass?"}
```

### Walk it step by step

1. **Create an immutable run identity.** Bind code commit, model revision, data split, environment, seed, and configuration before execution.
2. **Record the pruning trajectory.** Store per-stage sparsity, masks or retained indices, recovery checkpoints, and evaluation slices.
3. **Attach deployment evidence.** Keep export logs, runtime versions, operator traces, raw timing samples, and memory measurements with the same run.
4. **Reproduce before promotion.** A second run should rebuild the same candidate and reach tolerances defined in the manifest, not merely produce a similar headline metric.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 27
LESSON_TITLE = 'Automated Experiment Management and Reproducible Pruning Records'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260835
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | two executions with identical canonical configuration and seed |
| Candidate | one execution changing only the seed |
| Held constant | algorithm code, config schema, dimensions, target sparsity, dtype, hash method, and environment capture |
| Measurements | config hash, mask hash, output hash, sparsity, same-seed equality, and changed-seed difference |
| Evidence | `pytorch-gpu` |

**Experiment:** Run a pruning pipeline twice identically and once with a changed seed, then compare config, mask, and output hashes.


## 5. Read the experiment code

The notebook serializes configuration with sorted keys and compact separators before hashing. Masks move to CPU as contiguous bytes for a stable digest. The registry rows include environment and conclusion fields suitable for MLflow or W&B, but no external service is required to reproduce the core evidence.

Do not execute until the code implements the frozen table above.


In [2]:
def run(seed):
    torch.manual_seed(seed); config={"algorithm":"global_magnitude","shape":[256,256],"sparsity":0.75,"seed":seed,"dtype":"float32"}
    w=torch.randn(*config["shape"],device=DEVICE); x=torch.randn(16,256,device=DEVICE); mask=magnitude_mask(w,config["sparsity"]); out=F.linear(x,w*mask)
    canonical=json.dumps(config,sort_keys=True,separators=(",",":")); mask_bytes=mask.cpu().contiguous().numpy().tobytes(); out_bytes=out.detach().cpu().contiguous().numpy().tobytes()
    return {"config":config,"config_sha256":hashlib.sha256(canonical.encode()).hexdigest(),"mask_sha256":hashlib.sha256(mask_bytes).hexdigest(),"output_sha256":hashlib.sha256(out_bytes).hexdigest(),"sparsity":zero_fraction(mask)}
a=run(SEED); b=run(SEED); c=run(SEED+1)
metrics={"same_seed_config_match":a["config_sha256"]==b["config_sha256"],"same_seed_mask_match":a["mask_sha256"]==b["mask_sha256"],"same_seed_output_match":a["output_sha256"]==b["output_sha256"],"different_seed_mask_differs":a["mask_sha256"]!=c["mask_sha256"],"sparsity":a["sparsity"],"config_sha256":a["config_sha256"],"mask_sha256":a["mask_sha256"],"output_sha256":a["output_sha256"],"changed_seed_mask_sha256":c["mask_sha256"],"runs":[a,b,c]}
analysis=(f"Identical-seed runs matched config/mask/output hashes={metrics['same_seed_config_match']}/{metrics['same_seed_mask_match']}/"
          f"{metrics['same_seed_output_match']} at {metrics['sparsity']:.1%} sparsity. Changing only the seed changed the mask="
          f"{metrics['different_seed_mask_differs']}. The recorded support digest begins `{a['mask_sha256'][:12]}`.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Same-seed config match | yes |
| Same-seed mask match | yes |
| Same-seed output match | yes |
| Different-seed mask differs | yes |
| Sparsity | 75.00% |
| Mask SHA-256 | `89bdaa05855b` |


## 7. Interpret rather than merely print

Identical-seed runs matched config/mask/output hashes=True/True/True at 75.0% sparsity. Changing only the seed changed the mask=True. The recorded support digest begins `89bdaa05855b`.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 27,
    "title": 'Automated Experiment Management and Reproducible Pruning Records',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Reproducible pruning identifies the exact configuration, support, environment, and outputs—not merely the final zero percentage.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 27,
  "title": "Automated Experiment Management and Reproducible Pruning Records",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260835
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "same_seed_config_match": true,
    "same_seed_mask_match": true,
    "same_seed_output_match": true,
    "different_seed_mask_differs": true,
    "sparsity": 0.75,
    "config_sha256": "4182a833a49db2850e8ca57ccd0c54934e796ed460f629c65295fc8fd0f9f38e",
    "mask_sha256": "89bdaa05855b8278a2ac4048c61f9873abf7a3fadd0270289ffcf6db1330d986",
    "output_sha256": "a27663267d8b7954c593dbeb00adb21325781a67e931441fa2f10d74cc98e762",
    "changed_seed_mask_sha256": "8b615e8dd3a27d02b9b8f15a02554dbf4e6874e806367f0af960db41e1b70c60",
    "runs": [
      {
        "config": {
          "algorithm": "global_magnitude",
          "shape": [
            256,
   

## 9. Make the bounded decision

> Reproducible pruning identifies the exact configuration, support, environment, and outputs—not merely the final zero percentage.

**Acceptance/rollback:** Accept a reproduction claim only when an independent rerun matches the declared configuration, mask or bounded metrics, and environment-sensitive tolerances.

**Failure analysis:** Seeds do not guarantee bitwise equality across all devices, library versions, or nondeterministic kernels. Hashing only a filename or sparsity misses content changes.


## 10. Extend the evidence

Log the same schema to MLflow or W&B, rerun on a second machine, define which fields must match exactly versus within tolerance, and add checkpoint/export hashes.

The full evidence boundary and references are in [`README.md`](README.md).
